# Calendar data

## Data source
An external dataset that we build ourselves, from three parts:
- **Public holidays**: Python package [`holidays`](https://pypi.org/project/holidays/), `holidays.US(state="IL")` (US federal + Illinois).
- **Chicago Public Schools (CPS) breaks**: winter and spring break, entered by hand from the CPS academic calendars.
- **Days around holidays**: computed from the two parts above (eve, bridge days, long weekends, Thanksgiving week).

## Columns (`calendar.csv`, one row per day, 2022–2025)
| Column | Meaning |
|---|---|
| `date`, `year`, `month`, `day_of_week`, `dow_n` (0 = Monday), `is_weekend` | Basic time columns |
| `is_holiday`, `holiday_name` | Any Illinois holiday, and its name |
| `is_major_holiday` | New Year, Memorial Day, July 4, Labor Day, Thanksgiving, Christmas (with their "observed" days) |
| `is_new_year`, `is_memorial_day`, `is_july_4`, `is_labor_day`, `is_thanksgiving_day`, `is_christmas` | One flag per major holiday |
| `is_holiday_eve` | Working day just before a major holiday |
| `is_bridge_day` | Working day between a major holiday and the weekend (holiday on Tuesday → Monday; on Thursday → Friday) |
| `is_long_weekend` | Weekend days of a 3+ day block with a major holiday (days off anyway, **not** bridge days) |
| `is_thanksgiving_week` | Monday to Sunday of Thanksgiving week |
| `is_winter_break`, `is_spring_break` | CPS breaks |

## Objective
1. **Build** the calendar and check that the right days are flagged.
2. **Test** each calendar variable: do trips on those days differ, consistently, from comparable ordinary days?
3. **Choose** the flags for the model of a "normal" day, by adding them step by step to a negative binomial model.

All flags are kept in `calendar.csv`; only the ones with a clear effect go into the model.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import holidays            # listed in pip_requirements.txt

%matplotlib inline
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

### Settings

In [ ]:
START, END = "2022-01-01", "2025-12-31"   # study period
BUFFER_DAYS = 45                           # the calendar is built a bit beyond the study period: the flag of 31 Dec depends on 1 Jan
CITY_PATH = "data/daily_counts_city.csv"   # daily trips (date, n_trips, n_member, n_casual, n_electric)
SAVE_FOLDER = "data/external/"             # same folder as the other external datasets
os.makedirs(SAVE_FOLDER, exist_ok=True)

# 1. Building the calendar

### 1.1 Which holidays does the package know?
Every Illinois holiday of the study period, with its weekday.

In [ ]:
il = holidays.US(state="IL", years=range(pd.Timestamp(START).year - 1, pd.Timestamp(END).year + 2))
il_table = (pd.Series(il, name="holiday").rename_axis("date").reset_index()
              .assign(date=lambda d: pd.to_datetime(d.date)).sort_values("date"))
il_table = il_table[(il_table.date >= START) & (il_table.date <= END)]
il_table["weekday"] = il_table.date.dt.day_name()
il_table

Six holidays are days off for most people (New Year, Memorial Day, July 4, Labor Day, Thanksgiving, Christmas): we call them **major holidays**. The others (MLK Day, Presidents' Day, Lincoln's Birthday, Pulaski Day, Juneteenth, Columbus Day, Election Day, Veterans Day) are working days for most people; they stay in `is_holiday` and are tested in section 4.5.

### 1.2 Holiday flags
- **Observed days** are kept (e.g. "Christmas Day (observed)" on Monday 26 Dec 2022, when the 25th is a Sunday): people take that day off. Same rule every year.
- **One flag per major holiday**, because they do not behave alike (section 4.4).
- "Independence" also matches *Juneteenth National Independence Day*, so Juneteenth is excluded on purpose.

In [ ]:
MAJOR = {                      # flag name: text in holiday_name
    "is_new_year":         "New Year",
    "is_memorial_day":     "Memorial",
    "is_july_4":           "Independence Day",
    "is_labor_day":        "Labor",
    "is_thanksgiving_day": "Thanksgiving",
    "is_christmas":        "Christmas",
}

def build_calendar(start, end):
    cal = pd.DataFrame({"date": pd.date_range(start, end, freq="D")})
    il = holidays.US(state="IL", years=range(cal.date.dt.year.min(), cal.date.dt.year.max() + 1))
    d = cal.date.dt.date
    cal["is_holiday"] = d.isin(il)
    cal["holiday_name"] = d.map(il).fillna("")
    not_juneteenth = ~cal.holiday_name.str.contains("Juneteenth")
    for flag, text in MAJOR.items():
        cal[flag] = cal.holiday_name.str.contains(text) & not_juneteenth
    cal["is_major_holiday"] = cal[list(MAJOR)].any(axis=1)
    return cal

### 1.3 School breaks (Chicago Public Schools)
First and last day off of each break, entered by hand from the CPS calendars (the only hand-typed values of this notebook).

In [ ]:
SCHOOL_BREAKS = {
    "winter": [("2021-12-20", "2022-01-02"), ("2022-12-23", "2023-01-06"), ("2023-12-22", "2024-01-07"),
               ("2024-12-23", "2025-01-03"), ("2025-12-22", "2026-01-02")],
    "spring": [("2022-04-11", "2022-04-15"), ("2023-04-03", "2023-04-07"),
               ("2024-03-25", "2024-03-29"), ("2025-03-24", "2025-03-28")],
}

def add_school_breaks(cal):
    cal = cal.copy()
    for name, periods in SCHOOL_BREAKS.items():
        cal[f"is_{name}_break"] = False
        for s, e in periods:
            cal.loc[(cal.date >= s) & (cal.date <= e), f"is_{name}_break"] = True
    return cal

**Why only winter and spring break?**
- They are **multi-day blocks**, large enough to be measured.
- **Summer break** overlaps with the season, already captured by the annual sine/cosine terms.
- **Single days off** that are public holidays are already covered by the holiday flags.

### 1.4 Days around holidays
| Flag | Definition | Example |
|---|---|---|
| `is_holiday_eve` | Working day just before a major holiday | Wed 27 Nov 2024 (before Thanksgiving) |
| `is_bridge_day` | Working day between a major holiday and the weekend | Fri 29 Nov 2024 (after Thanksgiving) |
| `is_long_weekend` | Weekend days of a 3+ day block with a major holiday | Sat–Sun before Labor Day |
| `is_thanksgiving_week` | Monday to Sunday of Thanksgiving week | 25 Nov – 1 Dec 2024 |

A long-weekend day is a weekend day (people were off anyway); a bridge day is a working day that many people take off. They are different flags. A Monday before a Tuesday holiday is both an eve and a bridge day (e.g. Mon 3 Jul 2023).

In [ ]:
def add_holiday_context(cal):
    cal = cal.copy().sort_values("date").reset_index(drop=True)
    dow = cal.date.dt.dayofweek
    is_off = cal.is_major_holiday | (dow >= 5)
    working = ~is_off & ~cal.is_holiday                                  # ordinary working day

    # eve: working day just before a major holiday
    cal["is_holiday_eve"] = cal.is_major_holiday.shift(-1, fill_value=False) & working

    # bridge day: holiday on Tuesday -> the Monday before; holiday on Thursday -> the Friday after
    tue_hol = cal.is_major_holiday & (dow == 1)
    thu_hol = cal.is_major_holiday & (dow == 3)
    cal["is_bridge_day"] = (tue_hol.shift(-1, fill_value=False) | thu_hol.shift(1, fill_value=False)) & working

    # long weekend: non-holiday days of a block of 3+ consecutive days off that contains a major holiday
    block = (is_off != is_off.shift()).cumsum()
    block_len = is_off.groupby(block).transform("sum")
    has_hol = cal.is_major_holiday.groupby(block).transform("any")
    cal["is_long_weekend"] = is_off & (block_len >= 3) & has_hol & ~cal.is_major_holiday

    # Thanksgiving week: Monday to Sunday
    tg = cal.loc[cal.is_thanksgiving_day, "date"]
    week_starts = set(tg - pd.to_timedelta(tg.dt.dayofweek, unit="D"))
    cal["is_thanksgiving_week"] = (cal.date - pd.to_timedelta(dow, unit="D")).isin(week_starts)
    return cal

buffer = pd.Timedelta(days=BUFFER_DAYS)
calendar_full = add_holiday_context(add_school_breaks(build_calendar(pd.Timestamp(START) - buffer, pd.Timestamp(END) + buffer)))
print(f"Calendar built from {calendar_full.date.min().date()} to {calendar_full.date.max().date()} ({len(calendar_full)} days, including the buffer)")

### 1.5 Sanity checks

In [ ]:
# 1) one row per day, no gap, no duplicate, no NA
assert calendar_full.date.is_unique, "duplicated dates"
assert (calendar_full.date.diff().dropna() == pd.Timedelta(days=1)).all(), "gap in the dates"
assert not calendar_full.isna().any().any(), "NA values"
print("OK: one row per day, no gap, no duplicate, no NA")

In [ ]:
# 2) known dates must carry the right flags
checks = [
    ("2022-07-04", "is_july_4",           True,  "July 4th 2022"),
    ("2024-06-19", "is_july_4",           False, "Juneteenth 2024 is not July 4th"),
    ("2024-06-19", "is_holiday",          True,  "Juneteenth 2024 is a holiday..."),
    ("2024-06-19", "is_major_holiday",    False, "...but not a major one"),
    ("2023-05-29", "is_memorial_day",     True,  "Memorial Day 2023"),
    ("2025-09-01", "is_labor_day",        True,  "Labor Day 2025"),
    ("2024-11-28", "is_thanksgiving_day", True,  "Thanksgiving 2024"),
    ("2023-12-25", "is_christmas",        True,  "Christmas 2023"),
    ("2023-01-02", "is_new_year",         True,  "New Year 2023 observed (Monday)"),
    ("2024-03-12", "is_holiday",          False, "ordinary Tuesday"),
    ("2024-03-27", "is_spring_break",     True,  "spring break 2024"),
    ("2024-04-10", "is_spring_break",     False, "after spring break 2024"),
    ("2023-12-27", "is_winter_break",     True,  "winter break 2023"),
    ("2024-11-27", "is_thanksgiving_week", True, "Wednesday of Thanksgiving week 2024"),
    ("2024-11-27", "is_holiday_eve",      True,  "eve of Thanksgiving 2024"),
    ("2024-11-28", "is_holiday_eve",      False, "Thanksgiving 2024 is not an eve"),
    ("2024-12-24", "is_holiday_eve",      True,  "Christmas Eve 2024 (Tuesday)"),
    ("2025-12-31", "is_holiday_eve",      True,  "New Year's Eve 2025 (needs 1 Jan 2026: buffer)"),
    ("2023-12-31", "is_holiday_eve",      False, "New Year's Eve 2023 is a Sunday, not a working eve"),
    ("2022-05-29", "is_holiday_eve",      False, "Sunday before Memorial Day 2022 is not an eve"),
    ("2022-12-25", "is_holiday_eve",      False, "Christmas 2022 itself is not an eve"),
    ("2024-11-29", "is_bridge_day",       True,  "Friday after Thanksgiving 2024"),
    ("2023-07-03", "is_bridge_day",       True,  "Monday before July 4th 2023 (Tuesday)"),
    ("2024-07-05", "is_bridge_day",       True,  "Friday after July 4th 2024 (Thursday)"),
    ("2025-12-26", "is_bridge_day",       True,  "Friday after Christmas 2025 (Thursday)"),
    ("2025-07-03", "is_bridge_day",       False, "July 4th 2025 is a Friday: no bridge day"),
    ("2024-12-25", "is_long_weekend",     False, "Christmas Day 2024 is not a long weekend"),
    ("2025-08-30", "is_long_weekend",     True,  "Saturday before Labor Day 2025"),
]
cal_i = calendar_full.set_index("date")
chk = pd.DataFrame([{"date": d, "check": w, "column": c, "expected": e, "found": bool(cal_i.loc[d, c])}
                    for d, c, e, w in checks])
chk["ok"] = np.where(chk.expected == chk.found, "OK", "PROBLEM")
print(f"{(chk.ok == 'OK').sum()} / {len(chk)} checks passed")
chk

In [ ]:
# 3) every eve, bridge day and long-weekend day, with its weekday (to check by eye)
in_study = calendar_full[(calendar_full.date >= START) & (calendar_full.date <= END)]
for col in ["is_holiday_eve", "is_bridge_day", "is_long_weekend"]:
    d = in_study[in_study[col]]
    print(f"\n{col}: {len(d)} days")
    print(", ".join(d.date.dt.strftime("%Y-%m-%d %a")))
assert not (in_study[["is_holiday_eve", "is_bridge_day"]].any(axis=1) & (in_study.date.dt.dayofweek >= 5)).any(), \
    "an eve or bridge day falls on a weekend"
assert not (in_study.is_long_weekend & (in_study.date.dt.dayofweek < 5) & ~in_study.is_holiday).any(), \
    "a long-weekend day is an ordinary weekday"

*Eves and bridge days are all working days; long-weekend days are weekend days.*

In [ ]:
# 4) number of flagged days per year (a year with 0 of something is suspicious)
FLAGS_ALL = ["is_holiday", "is_major_holiday"] + list(MAJOR) + ["is_holiday_eve", "is_bridge_day", "is_long_weekend",
             "is_thanksgiving_week", "is_winter_break", "is_spring_break"]
in_study.assign(year=in_study.date.dt.year).groupby("year")[FLAGS_ALL].sum().T

*Each major holiday appears once a year, twice when an "observed" day is added (Christmas 2022, New Year 2023). Spring break = 5 days every year; Thanksgiving week = 7.*

### 1.6 Saving
The calendar is **clipped to the study period** before saving (the buffer was only needed to compute the flags at the edges).

In [ ]:
calendar = in_study.copy()
calendar["year"] = calendar.date.dt.year
calendar["month"] = calendar.date.dt.month
calendar["day_of_week"] = calendar.date.dt.day_name()
calendar["dow_n"] = calendar.date.dt.dayofweek
calendar["is_weekend"] = calendar.dow_n >= 5

ORDER = ["date", "year", "month", "day_of_week", "dow_n", "is_weekend", "is_holiday", "holiday_name",
         "is_major_holiday"] + list(MAJOR) + ["is_holiday_eve", "is_bridge_day", "is_long_weekend",
         "is_thanksgiving_week", "is_winter_break", "is_spring_break"]
calendar = calendar[ORDER].reset_index(drop=True)
calendar.to_csv(SAVE_FOLDER + "calendar.csv", index=False)
print(f"Saved {SAVE_FOLDER}calendar.csv: {len(calendar)} rows, {calendar.shape[1]} columns")
calendar.head()

# 2. What does the calendar look like?

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

calendar = pd.read_csv(SAVE_FOLDER + "calendar.csv", parse_dates=["date"])

def day_category(df):
    cat = np.zeros(len(df), dtype=int)                                                                    # 0 ordinary
    cat[(df.is_holiday_eve | df.is_bridge_day | df.is_long_weekend | df.is_thanksgiving_week).values] = 1  # around a holiday
    cat[(df.is_winter_break | df.is_spring_break).values] = 2                                              # school break
    cat[df.is_major_holiday.values] = 3                                                                    # major holiday
    return cat

COLORS = ["#ecebe7", "#e8b04a", "#7fa8d4", "#e34948"]
LABELS = ["ordinary day", "around a holiday (eve, bridge, long weekend, Thanksgiving week)", "school break", "major holiday"]
years = sorted(calendar.year.unique())
grid = np.full((len(years), 366), np.nan)
for i, y in enumerate(years):
    d = calendar[calendar.year == y]
    grid[i, d.date.dt.dayofyear.values - 1] = day_category(d)

fig, ax = plt.subplots(figsize=(15, 2.6))
ax.imshow(grid, aspect="auto", cmap=ListedColormap(COLORS), vmin=-.5, vmax=3.5, interpolation="nearest")
ax.set_yticks(range(len(years))); ax.set_yticklabels(years)
ax.set_xticks([pd.Timestamp(2023, m, 1).dayofyear - 1 for m in range(1, 13)])
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.grid(False)
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(COLORS, LABELS)], ncol=2, loc="upper center",
          bbox_to_anchor=(.5, -.18), frameon=False, fontsize=9)
ax.set_title("Calendar flags, one row per year", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

*Visual check: holidays on the right dates, breaks in winter and spring, flagged days next to holidays.*

In [ ]:
# School breaks: length and number of weekdays inside the study period
rows = []
for kind, periods in SCHOOL_BREAKS.items():
    for a, b in periods:
        d = calendar[(calendar.date >= a) & (calendar.date <= b)]
        rows.append({"break": kind, "first day": a, "last day": b, "days in study period": len(d),
                     "weekdays": int((~d.is_weekend).sum())})
pd.DataFrame(rows)

*The winter break of 2021–22 has only 2 days (weekend) inside the study period; the other breaks are complete.*

# 3. Integration with the trips

In [ ]:
daily_trips = pd.read_csv(CITY_PATH, parse_dates=["date"])
trips = daily_trips[(daily_trips.date >= START) & (daily_trips.date <= END)].copy()
missing_trip_days = pd.date_range(START, END).difference(trips.date)
print(f"{len(trips)} days of trips | days without trips data: {len(missing_trip_days)}")

merged = trips.merge(calendar, on="date", how="inner").sort_values("date").reset_index(drop=True)
assert len(merged) == len(calendar), "some calendar days have no trips"
merged.to_csv(SAVE_FOLDER + "dailyTrips_calendar.csv", index=False)
merged.head(3)

*Every day of 2022–2025 has both trips and a calendar row: the merge loses nothing.*

# 4. Does the calendar explain ridership?

For each calendar variable, three questions:
1. **Does it show on a plot?**
2. **Is the difference real?** (95 % confidence interval)
3. **Is it consistent?** (same sign every year)

### The "reference day"
A July 4th cannot be compared with the average day (it is in summer). So each day is compared with the **median of ordinary days with the same weekday within ±28 days** (ordinary = no holiday, no flag, no break):

> **ratio = trips / reference** — 1.00 = as usual, 0.60 = 40 % fewer trips than a normal day of that weekday at that time of year.

This uses the data only (no model).

In [ ]:
NOT_ORDINARY = ["is_holiday", "is_major_holiday", "is_holiday_eve", "is_bridge_day", "is_long_weekend",
                "is_thanksgiving_week", "is_winter_break", "is_spring_break"]

def local_reference(df, excluded, col="n_trips", window=28, min_days=3):
    t = (df.date - df.date.min()).dt.days.values
    dow, y, ok = df.dow_n.values, df[col].values.astype(float), ~excluded.values
    ref = np.full(len(df), np.nan)
    idx = np.arange(len(df))
    for i in range(len(df)):
        m = ok & (dow == dow[i]) & (np.abs(t - t[i]) <= window) & (idx != i)
        if m.sum() >= min_days:
            ref[i] = np.median(y[m])
    return ref

merged["ordinary"] = ~merged[NOT_ORDINARY].any(axis=1)
merged["ref"] = local_reference(merged, ~merged.ordinary)
merged["ratio"] = merged.n_trips / merged.ref
print(f"{int((~merged.ordinary).sum())} flagged days | median ratio of ordinary days = "
      f"{merged.loc[merged.ordinary, 'ratio'].median():.3f} (should be ~1)")

### Helper: effect table
For each group of days:
- **typical change** = median ratio − 1;
- **95 % confidence interval**: bootstrap that resamples whole *occurrences* (e.g. a whole spring break), because days of the same occurrence are not independent;
- **same side %**: share of days on the same side as the median;
- **clear effect?** = YES if the interval excludes 0.

With only 4 occurrences, "no" can also mean "not enough occurrences to tell".

In [ ]:
def occurrences(dates, gap_days=3):
    # days less than gap_days apart belong to the same occurrence (one break, one holiday weekend...)
    d = pd.Series(dates).sort_values()
    return (d.diff().dt.days.fillna(999) > gap_days).cumsum().reindex(pd.Series(dates).index).values

def cluster_bootstrap_ci(values, cluster, n_boot=4000, seed=0):
    rng = np.random.default_rng(seed)
    groups = [values[cluster == c] for c in np.unique(cluster)]
    meds = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(groups), len(groups))
        meds[b] = np.median(np.concatenate([groups[k] for k in pick]))
    return np.percentile(meds, [2.5, 97.5])

def effect_table(groups, df=None, ratio="ratio"):
    df = merged if df is None else df
    rows = []
    for label, mask in groups:
        sub = df.loc[mask & df[ratio].notna(), ["date", ratio]]
        if len(sub) == 0:
            continue
        r = sub[ratio].values
        cl = occurrences(sub.date)
        med = np.median(r) - 1
        lo, hi = cluster_bootstrap_ci(r, cl) if len(np.unique(cl)) > 1 else (np.nan, np.nan)
        same = np.mean((r - 1) * np.sign(med) > 0) if med != 0 else np.nan
        rows.append({"group": label, "days": len(r), "occurrences": len(np.unique(cl)), "typical change": med,
                     "CI low": lo - 1, "CI high": hi - 1, "share of days on same side": same})
    out = pd.DataFrame(rows)
    out["clear effect?"] = np.where(out["CI low"].isna(), "not enough occurrences",
                             np.where((out["CI low"] > 0) | (out["CI high"] < 0), "YES", "no"))
    return out

def fmt_effect(t):
    f = t.copy()
    for c in ["typical change", "CI low", "CI high", "share of days on same side"]:
        f[c] = (f[c] * 100).round(1)
    return f.rename(columns={"typical change": "change %", "CI low": "CI low %", "CI high": "CI high %",
                             "share of days on same side": "same side %"})

def effect_plot(groups, effects, title, df=None, ratio="ratio"):
    df = merged if df is None else df
    fig, ax = plt.subplots(figsize=(11, 0.5 * len(effects) + 1.8))
    ypos = np.arange(len(effects))[::-1]
    masks = dict(groups)
    for y, (_, row) in zip(ypos, effects.iterrows()):
        r = (df.loc[masks[row["group"]], ratio].dropna() - 1) * 100
        ax.barh(y, row["typical change"] * 100, height=.55, color="#2f6db5" if row["typical change"] < 0 else "#e34948", alpha=.85, zorder=2)
        ax.scatter(r, np.full(len(r), y), s=18, color="#8a8a8a", zorder=4)
        if not np.isnan(row["CI low"]):
            ax.errorbar(row["typical change"] * 100, y,
                        xerr=[[(row["typical change"] - row["CI low"]) * 100], [(row["CI high"] - row["typical change"]) * 100]],
                        fmt="none", ecolor="#1a1a1a", elinewidth=1.8, capsize=4, zorder=5)
        ax.text(1.01, y, f"{row['typical change'] * 100:+.0f} %  ({row['occurrences']} occ.)",
                transform=ax.get_yaxis_transform(), va="center", fontsize=9, color="#4d4d4d")
    ax.axvline(0, color="#4d4d4d", lw=1)
    ax.set_yticks(ypos); ax.set_yticklabels(effects.group)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.0f} %")); ax.grid(axis="y", visible=False)
    ax.set_xlabel("trips vs ordinary days of the same weekday nearby   (bar = median, black line = 95 % CI, grey dots = each day)")
    ax.set_title(title, loc="left", fontweight="bold")
    plt.subplots_adjust(right=.78, left=.3, bottom=.16); plt.show()

## 4.1 Weekday and weekend

In [ ]:
# ratio to the average of the same calendar week (removes season and year); complete weeks only
iso = merged.date.dt.isocalendar()
merged["iso_week"] = iso.year.astype(str) + "-" + iso.week.astype(str).str.zfill(2)
wk_mean = merged.groupby("iso_week").n_trips.transform("mean")
wk_count = merged.groupby("iso_week").n_trips.transform("count")
merged["week_ratio"] = np.where(wk_count == 7, merged.n_trips / wk_mean, np.nan)
merged["season"] = np.select([merged.month.isin([12, 1, 2]), merged.month.isin([6, 7, 8])],
                             ["Winter (Dec-Feb)", "Summer (Jun-Aug)"], "other")
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [ ]:
# Weekday vs weekend, all seasons together (ordinary days, complete weeks)
base = merged[merged.ordinary & merged.week_ratio.notna()].copy()
base["day type"] = np.where(base.is_weekend, "Weekend", "Weekday")

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(data=base, x="day type", y="week_ratio", order=["Weekday", "Weekend"], color="#d9e4f2", showfliers=False, ax=ax)
ax.axhline(1, color="gray", lw=1)
ax.set_title("Weekday vs weekend", loc="left", fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("trips / weekly average")
plt.tight_layout(); plt.show()

wkend, wkday = base[base.is_weekend].week_ratio, base[~base.is_weekend].week_ratio
print(f"Median ratio: weekday {wkday.median():.3f} vs weekend {wkend.median():.3f}  "
      f"(Mann-Whitney p = {stats.mannwhitneyu(wkend, wkday).pvalue:.2g})")

**Insights:** Over the whole year, weekdays and weekends have almost the same median, but weekends are much more spread out. This suggests two opposite behaviours mixed together, probably depending on the season → split by winter and summer.

In [ ]:
plot_df = merged[(merged.season != "other") & merged.ordinary & merged.week_ratio.notna()]
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=plot_df, x="day_of_week", y="week_ratio", hue="season", order=DAYS,
            palette={"Winter (Dec-Feb)": "#2f6db5", "Summer (Jun-Aug)": "#d98324"}, showfliers=False, ax=ax)
ax.axhline(1, color="gray", lw=1)
ax.set_title("Trips relative to the average day of the same week (ordinary days only)", loc="left", fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("trips / weekly average"); ax.legend(title="")
plt.tight_layout(); plt.show()

In [ ]:
# Median ratio of each weekday and season, with a 95 % bootstrap CI
rng = np.random.default_rng(0)
def boot_median_ci(values, n_boot=4000):
    meds = [np.median(rng.choice(values, len(values))) for _ in range(n_boot)]
    return np.percentile(meds, [2.5, 97.5])

rows = []
for (sea, d), g in plot_df.groupby(["season", "day_of_week"]):
    lo, hi = boot_median_ci(g.week_ratio.values)
    rows.append({"season": sea, "day": d, "median": g.week_ratio.median(), "CI low": lo, "CI high": hi, "days": len(g)})
wd_table = pd.DataFrame(rows)
wd_table["day"] = pd.Categorical(wd_table.day, DAYS, ordered=True)
wd_table = wd_table.sort_values(["season", "day"]).set_index(["season", "day"]).round(3)

# Is the weekend different in summer and winter? Resample whole WEEKS (Saturday and Sunday of a week are not independent)
we = plot_df[plot_df.is_weekend]
weeks = {s: [g.week_ratio.values for _, g in we[we.season.str.startswith(s)].groupby("iso_week")] for s in ["Summer", "Winter"]}
diffs = []
for _ in range(4000):
    summer = np.concatenate([weeks["Summer"][i] for i in rng.integers(0, len(weeks["Summer"]), len(weeks["Summer"]))])
    winter = np.concatenate([weeks["Winter"][i] for i in rng.integers(0, len(weeks["Winter"]), len(weeks["Winter"]))])
    diffs.append(np.median(summer) - np.median(winter))
lo, hi = np.percentile(diffs, [2.5, 97.5])
m_s, m_w = we[we.season.str.startswith("Summer")].week_ratio.median(), we[we.season.str.startswith("Winter")].week_ratio.median()
print(f"Weekend ratio: summer {m_s:.3f} vs winter {m_w:.3f}, difference {m_s - m_w:+.3f}, 95 % CI [{lo:+.3f}, {hi:+.3f}]")
wd_table

**Insights:** The weekly pattern is **reversed between seasons**. In summer, trips rise through the week and peak on Saturday; in winter, Tuesday–Thursday are the busiest and the weekend is the quietest. The weekend ratio is clearly higher in summer than in winter (CI excludes 0).
→ The model needs a **weekday** term **and** a **weekend × season** interaction.

## 4.2 Season

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=merged, x="month", y="n_trips", hue="year", palette="Blues", showfliers=False, ax=ax)
ax.set_title("Daily trips by month and year", loc="left", fontweight="bold")
ax.set_xlabel("Month"); ax.set_ylabel("trips per day"); ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.legend(title="Year", ncol=4)
plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
jul, jan = merged.loc[merged.month == 7, "n_trips"].values, merged.loc[merged.month == 1, "n_trips"].values
ratio_boot = [np.median(rng.choice(jul, len(jul))) / np.median(rng.choice(jan, len(jan))) for _ in range(5000)]
lo, hi = np.percentile(ratio_boot, [2.5, 97.5])
print(f"Median July / median January = {np.median(jul) / np.median(jan):.1f}x, 95 % CI [{lo:.1f}, {hi:.1f}]")

**Insights:** Season is the largest calendar effect (median July day ≈ 6× the median January day). The pattern is **one smooth hump** (peak July–August, low in January–February), with no jump between months, and the same shape every year.
→ A **smooth annual wave** (sine/cosine, 3 harmonics = 6 parameters) instead of 12 month effects.

## 4.3 Year

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
mm = merged.groupby(["year", "month"]).n_trips.mean().unstack(0)
for yr, c in zip(mm.columns, ["#bcdfea", "#7fa8d4", "#3f78b8", "#123f78"]):
    ax.plot(mm.index, mm[yr], lw=2.2, color=c, label=str(yr))
ax.set_xticks(range(1, 13)); ax.set_xlabel("Month"); ax.set_ylabel("average trips per day")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}")); ax.legend(title="Year")
ax.set_title("Average daily trips by month, one line per year", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Level of each year vs 2022, season removed (each day divided by the median day of its month, all years together)
merged["month_ratio"] = merged.n_trips / merged.groupby("month").n_trips.transform("median")
rng = np.random.default_rng(0)
first = merged.loc[merged.year == merged.year.min(), "month_ratio"].values
rows = []
for y, g in merged.groupby("year"):
    v = g.month_ratio.values
    boot = [np.median(rng.choice(v, len(v))) / np.median(rng.choice(first, len(first))) for _ in range(5000)]
    rows.append({"year": y, "level vs 2022": np.median(v) / np.median(first),
                 "CI low": np.percentile(boot, 2.5), "CI high": np.percentile(boot, 97.5)})
pd.DataFrame(rows).set_index("year").round(3)

**Insights:** The four years have the same seasonal shape at different heights. Relative to 2022, 2023 and 2024 are about 6–8 % higher (CI excludes 1), and 2025 is back to the 2022 level.
→ **One level per year** in the model. The gaps between years are not perfectly constant across months (larger in winter): this will remain in the residuals, probably weather.

## 4.4 Major holidays and days around them

In [ ]:
HOLIDAY_GROUPS = [
    ("New Year's Day",                 merged.is_new_year),
    ("Memorial Day",                   merged.is_memorial_day),
    ("July 4th",                       merged.is_july_4),
    ("Labor Day",                      merged.is_labor_day),
    ("Thanksgiving Day",               merged.is_thanksgiving_day),
    ("Christmas",                      merged.is_christmas),
    ("Eve of a holiday",               merged.is_holiday_eve),
    ("Bridge day",                     merged.is_bridge_day),
    ("Long weekend (weekend days)",    merged.is_long_weekend),
    ("Thanksgiving week (other days)", merged.is_thanksgiving_week & ~merged.is_thanksgiving_day & ~merged.is_bridge_day),
]
holiday_effects = effect_table(HOLIDAY_GROUPS)
effect_plot(HOLIDAY_GROUPS, holiday_effects, "Major holidays and days around them: how far from a normal day?")
fmt_effect(holiday_effects)

**How to read it.** A real effect = bar far from 0, all dots on the same side, black line (95 % CI) not crossing 0.

**Insights:** Holidays do not behave alike, so one pooled "major holiday" flag is not appropriate:
- **Clear effect:** Christmas (≈ −75 %), Thanksgiving Day (≈ −65 %), New Year (≈ −16 %), Labor Day (≈ +13 %, a leisure day at the end of summer), and the other days of Thanksgiving week.
- **No clear effect:** Memorial Day and July 4th (intervals include 0, 4 occurrences each), long weekends (interval narrow around 0: weekend days behave like normal weekends).
- **To check after running:** eve of a holiday, bridge days.

## 4.5 Minor holidays
Is it right to ignore them? They are working days for most people.

In [ ]:
nm = merged.holiday_name.fillna("")
MINOR = {"MLK Day": "Martin Luther King", "Presidents' Day": "Washington", "Lincoln's Birthday": "Lincoln",
         "Pulaski Day": "Pulaski", "Juneteenth": "Juneteenth", "Columbus Day": "Columbus",
         "Election Day": "Election", "Veterans Day": "Veterans"}
MINOR_GROUPS = [(label, merged.is_holiday & ~merged.is_major_holiday & ~merged.is_weekend & nm.str.contains(text))
                for label, text in MINOR.items()]
minor_effects = effect_table(MINOR_GROUPS)
effect_plot(MINOR_GROUPS, minor_effects, "Minor holidays (weekdays only)")
fmt_effect(minor_effects)

*Caveat: 8 holidays are tested with 4 occurrences each, so one "YES" can appear by chance. Only trust it if it is large, all dots are on the same side, and it makes sense.*

**Insights:** *(to fill after running: if no interval excludes 0 → minor holidays are not needed in the model; if one does, add it as a candidate flag)*

## 4.6 School breaks (CPS)

In [ ]:
def break_periods(kind):
    out = []
    for a, b in SCHOOL_BREAKS[kind]:
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        if ((merged.date >= a) & (merged.date <= b)).sum() >= 3:
            out.append((a, b))
    return out

fig, axes = plt.subplots(2, 4, figsize=(15, 6.2))
for r, kind in enumerate(["winter", "spring"]):
    per = break_periods(kind)
    for c in range(4):
        ax = axes[r, c]
        if c >= len(per):
            ax.axis("off"); continue
        a, b = per[c]
        w = merged[(merged.date >= a - pd.Timedelta(days=21)) & (merged.date <= b + pd.Timedelta(days=21))]
        wd, we = w[~w.is_weekend], w[w.is_weekend]
        ax.axvspan(a, b + pd.Timedelta(days=1), color="#d9e4f2", zorder=1)
        ax.plot(wd.date, wd.n_trips, color="#2f6db5", lw=1.6, marker="o", ms=3, zorder=3)
        ax.scatter(we.date, we.n_trips, s=14, color="#8a8a8a", zorder=3)
        ax.set_title(f"{kind.capitalize()} break {a:%b %Y}", fontsize=10)
        ax.xaxis.set_major_locator(plt.matplotlib.dates.WeekdayLocator(byweekday=0, interval=2))
        ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%d %b"))
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
        ax.set_ylim(bottom=0); ax.grid(axis="x", visible=False)
        if c == 0: ax.set_ylabel("trips per day")
fig.suptitle("School breaks (shaded): weekdays (blue line) vs weekends (grey dots), 3 weeks before and after",
             x=.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
wd = ~merged.is_weekend
winter_only = merged.is_winter_break & ~merged.is_major_holiday & ~merged.is_holiday_eve & ~merged.is_bridge_day
BREAK_GROUPS = [
    ("Winter break, weekdays (without holidays)", winter_only & wd),
    ("Winter break, weekends",                    winter_only & ~wd),
    ("Spring break, weekdays",                    merged.is_spring_break & wd),
]
break_effects = effect_table(BREAK_GROUPS)
fmt_effect(break_effects)

**How to read it.** A school/commute effect should show on **weekdays**. Winter break contains Christmas and New Year, so holidays, eves and bridge days are removed from it; spring break is the cleanest test.

**Insights:** Weekday trips drop during winter breaks, but this is mixed with the holidays and the cold of late December. No consistent drop during spring break.
→ Winter break is a **candidate**, to be tested in the model *with the holiday flags already in* (section 5). Spring break: no effect.

## 4.7 Members vs casual riders
A holiday can lower member trips (no commute) and raise casual trips (leisure): the two can cancel out in the total. Same method, one ratio per rider type.

In [ ]:
for col in ["n_member", "n_casual"]:
    merged[f"ratio_{col}"] = merged[col] / local_reference(merged, ~merged.ordinary, col=col)

rows = []
for col, label in [("ratio_n_member", "members"), ("ratio_n_casual", "casual")]:
    t = effect_table(HOLIDAY_GROUPS, ratio=col)[["group", "typical change", "CI low", "CI high", "clear effect?"]]
    rows.append(t.set_index("group").add_prefix(f"{label}: "))
rider_effects = pd.concat(rows, axis=1)
for c in rider_effects.columns:
    if "clear" not in c:
        rider_effects[c] = (rider_effects[c] * 100).round(1)
rider_effects

**Insights:** *(to fill after running: look for holidays where members and casual go in opposite directions, e.g. July 4th or Memorial Day; that would explain "no effect" in the total. If so, it is an argument for separate models per rider type in P3)*

## 4.8 Summary: candidate flags for the model

In [ ]:
summary = pd.concat([holiday_effects, minor_effects, break_effects], ignore_index=True)
fmt_effect(summary)[["group", "occurrences", "change %", "CI low %", "CI high %", "clear effect?"]]

| Decision | Flags |
|---|---|
| **In the model** (clear effect) | `is_christmas`, `is_thanksgiving_day`, `is_new_year`, `is_labor_day`, `is_thanksgiving_week` |
| **Candidates** (unclear: the model decides, section 5) | `is_holiday_eve`, `is_bridge_day`, `is_memorial_day` + `is_july_4`, `is_winter_break` |
| **Not in the model** (no effect, kept in `calendar.csv`) | `is_long_weekend`, `is_spring_break`, minor holidays, `is_major_holiday` (replaced by one flag per holiday) |

Plus the non-holiday terms justified in 4.1–4.3: weekday, weekend × season, smooth annual wave, year level.

# 5. From effects to the model

Section 4 looks at each variable alone. The question here: **once the others are in, does this variable still improve the description of a normal day?**

We add variables step by step to a **negative binomial model** and compare:
- **AIC / BIC**: lower is better (penalised for the number of parameters);
- **likelihood-ratio (LR) test** vs the previous model: is the improvement more than chance?
- **α** (leftover noise): a useful variable should lower it;
- **cross-validated error**: fit on 4/5 of the days (14-day blocks), predict the other 1/5, five times; median absolute % error. Shows whether a variable helps **out of sample** (no overfitting).

**Rule:** a candidate is kept if AIC goes down, the LR test is significant (p < 0.05) **and** the out-of-sample error does not go up.

In [ ]:
N_HARMONICS = 3
BOOL_FLAGS = list(MAJOR) + ["is_major_holiday", "is_holiday_eve", "is_bridge_day", "is_long_weekend",
                            "is_thanksgiving_week", "is_winter_break", "is_spring_break"]

def add_features(df):
    df = df.copy()
    doy = df.date.dt.dayofyear
    for k in range(1, N_HARMONICS + 1):
        df[f"sin{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
        df[f"cos{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
    df["wkend_sin1"] = df.is_weekend.astype(int) * df.sin1
    df["wkend_cos1"] = df.is_weekend.astype(int) * df.cos1
    for c in BOOL_FLAGS:
        df[c] = df[c].astype(int)
    df["year_c"] = df.year.astype(str)
    return df

md_df = add_features(merged)

def fit_nb(formula, data):
    return smf.negativebinomial(formula, data).fit(disp=False, maxiter=500, method="bfgs")

K, BLOCK = 5, 14
fold = (np.arange(len(md_df)) // BLOCK) % K     # 14-day blocks: neighbouring days stay in the same fold
def cv_error(formula):
    errs = []
    for k in range(K):
        train, test = md_df[fold != k], md_df[fold == k]
        mu = fit_nb(formula, train).predict(test)
        errs.append(np.abs(test.n_trips.values - mu.values) / test.n_trips.values)
    return np.median(np.concatenate(errs)) * 100

def compare(models):
    # models: list of (name, formula), each nested in the previous one (or in the one given as base)
    rows, fits = [], {}
    for name_, fo, base in models:
        res = fit_nb(fo, md_df); fits[name_] = res
        row = {"model": name_, "parameters": len(res.params), "AIC": res.aic, "BIC": res.bic, "alpha": res.params["alpha"]}
        if base is not None:
            prev = fits[base]
            lr = 2 * (res.llf - prev.llf)
            row["vs"] = base.split()[0]
            row["ΔAIC"] = res.aic - prev.aic
            row["p (LR test)"] = stats.chi2.sf(lr, len(res.params) - len(prev.params))
        row["CV error %"] = cv_error(fo)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model"), fits

### 5.1 Core model: season, year, weekday, holidays

In [ ]:
SEASON = " + ".join(f"sin{k} + cos{k}" for k in range(1, N_HARMONICS + 1))
HOLIDAYS = "is_christmas + is_thanksgiving_day + is_new_year + is_labor_day + is_thanksgiving_week"
F0 = f"n_trips ~ C(year_c) + {SEASON}"
F1 = F0 + " + C(day_of_week)"
F2 = F1 + " + wkend_sin1 + wkend_cos1"
F3 = F2 + " + " + HOLIDAYS
core_table, core_fits = compare([
    ("M0  year + season",        F0, None),
    ("M1  + weekday",            F1, "M0  year + season"),
    ("M2  + weekend x season",   F2, "M1  + weekday"),
    ("M3  + holiday flags",      F3, "M2  + weekend x season"),
])
core_table.round({"AIC": 1, "BIC": 1, "alpha": 4, "ΔAIC": 1, "CV error %": 2})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
labels = [m.split("  ")[0] for m in core_table.index]
for ax, col, title in zip(axes, ["AIC", "alpha", "CV error %"],
                          ["AIC (lower = better)", "leftover noise α (lower = more explained)", "out-of-sample error, % (lower = better)"]):
    ax.bar(labels, core_table[col], color="#7fa8d4", width=.6)
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold"); ax.grid(axis="x", visible=False)
    lo, hi = core_table[col].min(), core_table[col].max()
    ax.set_ylim(lo - (hi - lo) * 0.6 - 1e-9, hi + (hi - lo) * 0.15 + 1e-9)       # zoomed: only differences matter
    for i, v in enumerate(core_table[col]):
        ax.text(i, v, f"{v:,.0f}" if col == "AIC" else f"{v:.4f}" if col == "alpha" else f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

**Insights:** *(to check after running — in Laure's version: each step lowers AIC and α; weekday alone does not help out of sample, only together with weekend × season, because the weekly pattern is reversed between winter and summer; holidays bring the largest improvement)*

### 5.2 One flag per holiday vs one pooled flag
Check that separate holiday flags describe the data better than the old pooled `is_major_holiday`.

In [ ]:
F3_pooled = F2 + " + is_major_holiday + is_christmas + is_thanksgiving_week"
pooled_table, _ = compare([("M3 pooled  is_major_holiday + Christmas + Thanksgiving week", F3_pooled, None),
                           ("M3 separate  one flag per holiday", F3, None)])
pooled_table[["parameters", "AIC", "BIC", "alpha", "CV error %"]].round({"AIC": 1, "BIC": 1, "alpha": 4, "CV error %": 2})

*Lower AIC and error for the separate flags = the pooled flag mixed holidays with opposite effects.*

### 5.3 Candidate flags
Each candidate is added **alone** to M3, so its effect is not mixed with the others.

In [ ]:
CANDIDATES = {
    "holiday eve":            "is_holiday_eve",
    "bridge day":             "is_bridge_day",
    "Memorial Day + July 4":  "is_memorial_day + is_july_4",
    "winter break":           "is_winter_break",
    "spring break":           "is_spring_break",
}
cand_models = [("M3  core model", F3, None)] + [(f"M3 + {k}", F3 + " + " + v, "M3  core model") for k, v in CANDIDATES.items()]
cand_table, cand_fits = compare(cand_models)
cv_core = cand_table.loc["M3  core model", "CV error %"]
cand_table["keep?"] = np.where(cand_table.index == "M3  core model", "",
                       np.where((cand_table["ΔAIC"] < 0) & (cand_table["p (LR test)"] < 0.05) & (cand_table["CV error %"] <= cv_core),
                                "YES", "no"))
cand_table[["ΔAIC", "p (LR test)", "alpha", "CV error %", "keep?"]].round({"ΔAIC": 1, "alpha": 4, "CV error %": 2})

**Insights:** *(to fill after running: which candidates pass the rule, and why the others don't)*

### 5.4 Final model

In [ ]:
kept = [CANDIDATES[k] for k in CANDIDATES if cand_table.loc[f"M3 + {k}", "keep?"] == "YES"]
F_FINAL = F3 + "".join(" + " + v for v in kept)
MODEL_FLAGS = HOLIDAYS.split(" + ") + [f for v in kept for f in v.split(" + ")]
print("Calendar flags in the final model:", MODEL_FLAGS)

final = fit_nb(F_FINAL, md_df)
ci = np.exp(final.conf_int())
effects = pd.DataFrame({"multiplier": np.exp(final.params), "CI low": ci[0], "CI high": ci[1]}).loc[MODEL_FLAGS]
effects["change %"] = ((effects.multiplier - 1) * 100).round(1)
effects.round(3)

*Effect of each calendar flag in the final model, all other terms held fixed (multiplier 0.31 = −69 % trips). These flags are the ones used in the model of a "normal" day.*

# 6. Next steps (P3)
- Fit the model per cell (37 cells), with the flags chosen here.
- Detect anomalies: observed vs expected trips, negative binomial p-values, Benjamini–Hochberg.
- Check whether holidays with no city-wide effect (July 4th, Memorial Day) have a **local** effect (lakefront cells).